# Factor Model: Step 2 - Estimation Universe Selection

## Paleologo's Framework: Step 2 of 6

**Objective**: Select a subset of securities that are liquid, tradeable, and have sufficient data quality for factor estimation.

### Selection Criteria (Institutional Standards):
1. **Minimum Price**: $10.00 (exclude low-priced stocks)
2. **Minimum ADDV**: $2,000,000 (Average Daily Dollar Volume)
3. **Lookback Period**: 126 trading days (~6 months)
4. **Minimum Market Cap**: $100,000,000
5. **Data Quality**: Stocks must have price/volume data for lookback period

### Data Source:
- **Input**: russell2000_data_2005_2025.parquet (from Step 1)
- **Output**: russell2000_estimation_universe_step2.parquet
- **Format**: Panel data (MultiIndex: date × symbol)

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: '%.4f' % x)

# Plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print("Libraries imported successfully")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

## 2.1 Load Data

In [ ]:
# Load the Russell 2000 data
data_path = 'russell2000_data_2005_2025.parquet'

print("Loading data...")
df_raw = pd.read_parquet(data_path)

print(f"✓ Data loaded successfully")
print(f"Raw shape: {df_raw.shape}")
print(f"Columns: {df_raw.columns.tolist()}")
print(f"\nFirst few rows:")
display(df_raw.head())

In [ ]:
# Standardize column names to lowercase
print("Standardizing column names...")
df_raw = df_raw.rename(columns={
    'Symbol': 'symbol',
    'Date': 'date',
    'Open': 'open',
    'High': 'high',
    'Low': 'low',
    'Close': 'close',
    'Volume': 'volume',
    'GICS_Sector': 'gics_sector',
    'GICS_Industry_Group': 'gics_industry_group',
    'GICS_Industry': 'gics_industry',
    'GICS_SubIndustry': 'gics_subindustry'
})

# Ensure date is datetime
df_raw['date'] = pd.to_datetime(df_raw['date'])

print(f"✓ Column names standardized")
print(f"Columns: {df_raw.columns.tolist()}")

## 2.2 Define Universe Selection Criteria

In [ ]:
# Universe selection parameters (Institutional standards)
MIN_PRICE = 10.00                    # Minimum closing price ($10)
MIN_ADDV = 2_000_000                 # Minimum Average Daily Dollar Volume ($2M)
LOOKBACK_DAYS = 126                  # 6 months (~126 trading days)
MIN_MARKET_CAP = 100_000_000         # Minimum market cap ($100M)
MIN_DATA_POINTS = int(LOOKBACK_DAYS * 0.8)  # At least 80% of lookback period

print("="*80)
print("UNIVERSE SELECTION CRITERIA")
print("="*80)
print(f"\n1. Minimum Price: ${MIN_PRICE:,.2f}")
print(f"2. Minimum ADDV: ${MIN_ADDV:,.0f}")
print(f"3. Lookback Period: {LOOKBACK_DAYS} trading days (~6 months)")
print(f"4. Minimum Market Cap: ${MIN_MARKET_CAP:,.0f}")
print(f"5. Minimum Data Points: {MIN_DATA_POINTS} days ({MIN_DATA_POINTS/LOOKBACK_DAYS*100:.0f}% of lookback)")
print("\n" + "="*80)

## 2.3 Calculate Daily Dollar Volume

In [ ]:
print("Calculating daily dollar volume...")

# Calculate daily dollar volume (close price × volume)
df_raw['dollar_volume'] = df_raw['close'] * df_raw['volume']

print(f"✓ Daily dollar volume calculated")
print(f"\nDollar Volume Statistics:")
print(df_raw['dollar_volume'].describe())

# Show distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram (log scale)
axes[0].hist(np.log10(df_raw['dollar_volume'].replace(0, np.nan).dropna()), bins=50, color='steelblue', edgecolor='black')
axes[0].set_title('Daily Dollar Volume Distribution (log10 scale)', fontweight='bold')
axes[0].set_xlabel('log10(Dollar Volume)')
axes[0].set_ylabel('Frequency')
axes[0].axvline(np.log10(MIN_ADDV), color='red', linestyle='--', linewidth=2, label=f'${MIN_ADDV/1e6:.1f}M threshold')
axes[0].legend()

# Box plot (log scale)
axes[1].boxplot(np.log10(df_raw['dollar_volume'].replace(0, np.nan).dropna()))
axes[1].set_title('Daily Dollar Volume - Box Plot (log10 scale)', fontweight='bold')
axes[1].set_ylabel('log10(Dollar Volume)')
axes[1].axhline(np.log10(MIN_ADDV), color='red', linestyle='--', linewidth=2, label=f'${MIN_ADDV/1e6:.1f}M threshold')
axes[1].legend()

plt.tight_layout()
plt.show()

## 2.4 Calculate Rolling Average Daily Dollar Volume (ADDV)

In [ ]:
print(f"Calculating {LOOKBACK_DAYS}-day rolling ADDV for each stock...")
print("This may take a few minutes...\n")

# Sort by symbol and date
df_sorted = df_raw.sort_values(['symbol', 'date']).copy()

# Calculate rolling ADDV for each symbol
df_sorted['addv'] = df_sorted.groupby('symbol')['dollar_volume'].transform(
    lambda x: x.rolling(window=LOOKBACK_DAYS, min_periods=MIN_DATA_POINTS).mean()
)

# Calculate rolling average price for market cap calculation later
df_sorted['avg_price'] = df_sorted.groupby('symbol')['close'].transform(
    lambda x: x.rolling(window=LOOKBACK_DAYS, min_periods=MIN_DATA_POINTS).mean()
)

print(f"✓ Rolling ADDV calculated")
print(f"\nADDV Statistics:")
print(df_sorted['addv'].describe())

# Count rows with valid ADDV
valid_addv = df_sorted['addv'].notna().sum()
print(f"\nRows with valid ADDV: {valid_addv:,} ({valid_addv/len(df_sorted)*100:.2f}%)")

## 2.5 Calculate Market Capitalization

Note: True market cap = shares outstanding × price. Since we don't have shares outstanding data, we'll use a proxy:
- **Market Cap Proxy = (ADDV / Volume) × Shares Outstanding**
- We'll estimate shares outstanding from typical turnover rates
- Alternatively, we can use ADDV as a direct liquidity measure (which correlates with market cap)

For this implementation, we'll use **volume-weighted price × volume** as a market cap proxy.

In [ ]:
print("Calculating market cap proxy...")
print("Note: Without shares outstanding data, we'll use ADDV as market cap proxy")
print("      (Higher ADDV strongly correlates with higher market cap)\n")

# For proper market cap, we'd need shares outstanding
# As a proxy: stocks with consistently high ADDV over 6 months have higher market cap
# We'll use a simplified rule: Market Cap Proxy = ADDV × 30 (assuming ~3.3% daily turnover)
# This is rough but captures the relative sizing

df_sorted['market_cap_proxy'] = df_sorted['addv'] * 30

print(f"✓ Market cap proxy calculated")
print(f"\nMarket Cap Proxy Statistics:")
print(df_sorted['market_cap_proxy'].describe())

print("\n⚠ WARNING: This is a rough proxy. Ideally use actual shares outstanding.")
print("   For now, we'll rely primarily on price and ADDV filters.")

## 2.6 Apply Universe Selection Filters

In [ ]:
print("="*80)
print("APPLYING UNIVERSE SELECTION FILTERS")
print("="*80)

# Start with all data
df_filtered = df_sorted.copy()
initial_count = len(df_filtered)
initial_symbols = df_filtered['symbol'].nunique()

print(f"\nInitial dataset:")
print(f"  Rows: {initial_count:,}")
print(f"  Unique symbols: {initial_symbols:,}")
print(f"  Date range: {df_filtered['date'].min()} to {df_filtered['date'].max()}")

# Filter 1: Minimum Price
print(f"\n1. Applying minimum price filter (>= ${MIN_PRICE:.2f})...")
before = len(df_filtered)
df_filtered = df_filtered[df_filtered['close'] >= MIN_PRICE]
after = len(df_filtered)
removed = before - after
print(f"   Removed: {removed:,} rows ({removed/before*100:.2f}%)")
print(f"   Remaining: {after:,} rows")
print(f"   Symbols remaining: {df_filtered['symbol'].nunique():,}")

# Filter 2: Valid ADDV (must have ADDV calculated)
print(f"\n2. Applying valid ADDV filter (must have {MIN_DATA_POINTS}+ data points)...")
before = len(df_filtered)
df_filtered = df_filtered[df_filtered['addv'].notna()]
after = len(df_filtered)
removed = before - after
print(f"   Removed: {removed:,} rows ({removed/before*100:.2f}%)")
print(f"   Remaining: {after:,} rows")
print(f"   Symbols remaining: {df_filtered['symbol'].nunique():,}")

# Filter 3: Minimum ADDV
print(f"\n3. Applying minimum ADDV filter (>= ${MIN_ADDV:,.0f})...")
before = len(df_filtered)
df_filtered = df_filtered[df_filtered['addv'] >= MIN_ADDV]
after = len(df_filtered)
removed = before - after
print(f"   Removed: {removed:,} rows ({removed/before*100:.2f}%)")
print(f"   Remaining: {after:,} rows")
print(f"   Symbols remaining: {df_filtered['symbol'].nunique():,}")

# Filter 4: Minimum Market Cap (using proxy)
print(f"\n4. Applying minimum market cap proxy filter (>= ${MIN_MARKET_CAP:,.0f})...")
before = len(df_filtered)
df_filtered = df_filtered[df_filtered['market_cap_proxy'] >= MIN_MARKET_CAP]
after = len(df_filtered)
removed = before - after
print(f"   Removed: {removed:,} rows ({removed/before*100:.2f}%)")
print(f"   Remaining: {after:,} rows")
print(f"   Symbols remaining: {df_filtered['symbol'].nunique():,}")

# Final summary
final_count = len(df_filtered)
final_symbols = df_filtered['symbol'].nunique()
total_removed = initial_count - final_count

print("\n" + "="*80)
print("FILTERING SUMMARY")
print("="*80)
print(f"\nInitial: {initial_count:,} rows, {initial_symbols:,} symbols")
print(f"Final:   {final_count:,} rows, {final_symbols:,} symbols")
print(f"\nRemoved: {total_removed:,} rows ({total_removed/initial_count*100:.2f}%)")
print(f"Removed: {initial_symbols - final_symbols:,} symbols ({(initial_symbols - final_symbols)/initial_symbols*100:.2f}%)")
print("\n" + "="*80)

## 2.7 Universe Statistics and Visualization

In [ ]:
print("="*80)
print("ESTIMATION UNIVERSE STATISTICS")
print("="*80)

# Basic statistics
print(f"\n1. Data Coverage:")
print(f"   Total rows: {len(df_filtered):,}")
print(f"   Unique symbols: {df_filtered['symbol'].nunique():,}")
print(f"   Unique dates: {df_filtered['date'].nunique():,}")
print(f"   Date range: {df_filtered['date'].min()} to {df_filtered['date'].max()}")
print(f"   Average symbols per day: {len(df_filtered) / df_filtered['date'].nunique():.0f}")

# Price statistics
print(f"\n2. Price Statistics:")
print(df_filtered['close'].describe())

# ADDV statistics
print(f"\n3. ADDV Statistics:")
print(df_filtered['addv'].describe())

# Market cap proxy statistics
print(f"\n4. Market Cap Proxy Statistics:")
print(df_filtered['market_cap_proxy'].describe())

In [ ]:
# Universe size over time
print("\n5. Universe Size Over Time:")

symbols_per_month = df_filtered.groupby(pd.Grouper(key='date', freq='M'))['symbol'].nunique()

fig, ax = plt.subplots(figsize=(14, 6))
symbols_per_month.plot(ax=ax, color='green', linewidth=2)
ax.set_title('Estimation Universe Size Over Time', fontsize=14, fontweight='bold')
ax.set_xlabel('Date', fontsize=12)
ax.set_ylabel('Number of Stocks', fontsize=12)
ax.axhline(symbols_per_month.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {symbols_per_month.mean():.0f}')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"   Min universe size: {symbols_per_month.min():.0f} stocks")
print(f"   Max universe size: {symbols_per_month.max():.0f} stocks")
print(f"   Mean universe size: {symbols_per_month.mean():.0f} stocks")
print(f"   Std universe size: {symbols_per_month.std():.0f} stocks")

In [ ]:
# Distribution of filtered variables
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Price distribution
axes[0, 0].hist(df_filtered['close'], bins=50, color='steelblue', edgecolor='black')
axes[0, 0].set_title('Price Distribution', fontweight='bold')
axes[0, 0].set_xlabel('Close Price ($)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].axvline(MIN_PRICE, color='red', linestyle='--', linewidth=2, label=f'Min: ${MIN_PRICE}')
axes[0, 0].legend()

# ADDV distribution (log scale)
axes[0, 1].hist(np.log10(df_filtered['addv']), bins=50, color='coral', edgecolor='black')
axes[0, 1].set_title('ADDV Distribution (log10 scale)', fontweight='bold')
axes[0, 1].set_xlabel('log10(ADDV)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].axvline(np.log10(MIN_ADDV), color='red', linestyle='--', linewidth=2, label=f'Min: ${MIN_ADDV/1e6:.1f}M')
axes[0, 1].legend()

# Volume distribution (log scale)
axes[1, 0].hist(np.log10(df_filtered['volume'].replace(0, np.nan).dropna()), bins=50, color='lightgreen', edgecolor='black')
axes[1, 0].set_title('Volume Distribution (log10 scale)', fontweight='bold')
axes[1, 0].set_xlabel('log10(Volume)')
axes[1, 0].set_ylabel('Frequency')

# Market cap proxy distribution (log scale)
axes[1, 1].hist(np.log10(df_filtered['market_cap_proxy']), bins=50, color='plum', edgecolor='black')
axes[1, 1].set_title('Market Cap Proxy Distribution (log10 scale)', fontweight='bold')
axes[1, 1].set_xlabel('log10(Market Cap Proxy)')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].axvline(np.log10(MIN_MARKET_CAP), color='red', linestyle='--', linewidth=2, label=f'Min: ${MIN_MARKET_CAP/1e6:.0f}M')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

## 2.8 Sector Distribution in Estimation Universe

In [ ]:
print("="*80)
print("SECTOR DISTRIBUTION")
print("="*80)

# Get unique symbols and their sectors (most recent classification)
latest_date = df_filtered['date'].max()
latest_universe = df_filtered[df_filtered['date'] == latest_date].copy()

if 'gics_sector' in latest_universe.columns:
    sector_counts = latest_universe['gics_sector'].value_counts()
    
    print(f"\nSector Distribution (as of {latest_date.date()}):")
    print("\n{:<40s} {:>10s} {:>10s}".format('Sector', 'Count', 'Percentage'))
    print("-" * 62)
    for sector, count in sector_counts.items():
        pct = count / len(latest_universe) * 100
        print("{:<40s} {:>10d} {:>9.2f}%".format(str(sector), count, pct))
    
    # Plot sector distribution
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Bar chart
    sector_counts.plot(kind='bar', ax=axes[0], color='steelblue', edgecolor='black')
    axes[0].set_title('Sector Distribution - Bar Chart', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Sector', fontsize=12)
    axes[0].set_ylabel('Number of Stocks', fontsize=12)
    axes[0].tick_params(axis='x', rotation=45, labelsize=10)
    
    # Pie chart
    sector_counts.plot(kind='pie', ax=axes[1], autopct='%1.1f%%', startangle=90)
    axes[1].set_title('Sector Distribution - Pie Chart', fontsize=14, fontweight='bold')
    axes[1].set_ylabel('')
    
    plt.tight_layout()
    plt.show()
else:
    print("\n⚠ GICS sector information not available")

## 2.9 Universe Turnover Analysis

Analyze how many stocks enter/exit the estimation universe over time

In [ ]:
print("="*80)
print("UNIVERSE TURNOVER ANALYSIS")
print("="*80)

# Get monthly universe membership
df_filtered['year_month'] = df_filtered['date'].dt.to_period('M')
monthly_universe = df_filtered.groupby('year_month')['symbol'].apply(set)

# Calculate monthly additions and deletions
additions = []
deletions = []
turnover_pct = []

for i in range(1, len(monthly_universe)):
    prev_universe = monthly_universe.iloc[i-1]
    curr_universe = monthly_universe.iloc[i]
    
    added = len(curr_universe - prev_universe)
    deleted = len(prev_universe - curr_universe)
    total_change = added + deleted
    turnover = (total_change / len(prev_universe)) * 100 if len(prev_universe) > 0 else 0
    
    additions.append(added)
    deletions.append(deleted)
    turnover_pct.append(turnover)

print(f"\nMonthly Universe Turnover Statistics:")
print(f"  Mean additions per month: {np.mean(additions):.1f} stocks")
print(f"  Mean deletions per month: {np.mean(deletions):.1f} stocks")
print(f"  Mean turnover per month: {np.mean(turnover_pct):.2f}%")
print(f"  Max turnover per month: {np.max(turnover_pct):.2f}%")

# Plot turnover over time
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Additions and deletions
x = monthly_universe.index[1:]
axes[0].plot(x.to_timestamp(), additions, label='Additions', color='green', linewidth=2)
axes[0].plot(x.to_timestamp(), deletions, label='Deletions', color='red', linewidth=2)
axes[0].set_title('Monthly Universe Changes: Additions vs Deletions', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Date', fontsize=12)
axes[0].set_ylabel('Number of Stocks', fontsize=12)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Turnover percentage
axes[1].plot(x.to_timestamp(), turnover_pct, color='purple', linewidth=2)
axes[1].set_title('Monthly Universe Turnover %', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Date', fontsize=12)
axes[1].set_ylabel('Turnover %', fontsize=12)
axes[1].axhline(np.mean(turnover_pct), color='red', linestyle='--', linewidth=2, label=f'Mean: {np.mean(turnover_pct):.2f}%')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2.10 Final Data Quality Checks

In [ ]:
print("="*80)
print("FINAL DATA QUALITY CHECKS")
print("="*80)

# Missing data check
print("\n1. Missing Data:")
missing_summary = df_filtered.isnull().sum()
missing_pct = (missing_summary / len(df_filtered)) * 100
missing_df = pd.DataFrame({
    'Missing_Count': missing_summary,
    'Missing_Pct': missing_pct
}).sort_values('Missing_Pct', ascending=False)
print(missing_df[missing_df['Missing_Count'] > 0])

if missing_df['Missing_Count'].sum() == 0:
    print("   ✓ No missing data in core fields")

# OHLC consistency check
print("\n2. OHLC Consistency:")
invalid_hl = (df_filtered['high'] < df_filtered['low']).sum()
invalid_ho = (df_filtered['high'] < df_filtered['open']).sum()
invalid_hc = (df_filtered['high'] < df_filtered['close']).sum()
invalid_lo = (df_filtered['low'] > df_filtered['open']).sum()
invalid_lc = (df_filtered['low'] > df_filtered['close']).sum()

print(f"   High < Low: {invalid_hl:,}")
print(f"   High < Open: {invalid_ho:,}")
print(f"   High < Close: {invalid_hc:,}")
print(f"   Low > Open: {invalid_lo:,}")
print(f"   Low > Close: {invalid_lc:,}")

if invalid_hl + invalid_ho + invalid_hc + invalid_lo + invalid_lc == 0:
    print("   ✓ All OHLC relationships are consistent")

# Universe coverage
print("\n3. Universe Coverage:")
days_per_symbol = df_filtered.groupby('symbol')['date'].count()
print(f"   Mean days per symbol: {days_per_symbol.mean():.0f}")
print(f"   Median days per symbol: {days_per_symbol.median():.0f}")
print(f"   Min days per symbol: {days_per_symbol.min():.0f}")
print(f"   Max days per symbol: {days_per_symbol.max():.0f}")

# Symbols with < 250 days (less than 1 year)
short_history = (days_per_symbol < 250).sum()
print(f"   Symbols with < 1 year history: {short_history:,} ({short_history/len(days_per_symbol)*100:.2f}%)")

## 2.11 Convert to Panel Format and Save

In [ ]:
print("="*80)
print("PREPARING OUTPUT DATA")
print("="*80)

# Select columns for output
output_cols = ['symbol', 'date', 'open', 'high', 'low', 'close', 'volume', 
               'dollar_volume', 'addv', 'avg_price', 'market_cap_proxy']

# Add GICS columns if available
gics_cols = ['gics_sector', 'gics_industry_group', 'gics_industry', 'gics_subindustry']
for col in gics_cols:
    if col in df_filtered.columns:
        output_cols.append(col)

df_output = df_filtered[output_cols].copy()

# Convert to MultiIndex panel format
print("\nConverting to MultiIndex panel format (date, symbol)...")
df_output = df_output.set_index(['date', 'symbol']).sort_index()

print(f"✓ Data converted to panel format")
print(f"  Shape: {df_output.shape}")
print(f"  Index levels: {df_output.index.names}")
print(f"  Columns: {df_output.columns.tolist()}")

print("\nFirst few rows:")
display(df_output.head(10))

In [ ]:
# Save to file
output_path = 'russell2000_estimation_universe_step2.parquet'
print(f"\nSaving estimation universe to {output_path}...")

df_output.to_parquet(output_path, compression='snappy')

import os
file_size_mb = os.path.getsize(output_path) / 1024**2

print(f"✓ Data saved successfully!")
print(f"  File: {output_path}")
print(f"  Size: {file_size_mb:.2f} MB")
print(f"  Rows: {len(df_output):,}")
print(f"  Columns: {len(df_output.columns)}")

## 2.12 Summary Report

In [ ]:
print("="*80)
print("ESTIMATION UNIVERSE SELECTION - SUMMARY REPORT")
print("="*80)

print(f"\nReport Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

print("\n" + "="*80)
print("SELECTION CRITERIA")
print("="*80)
print(f"\n1. Minimum Price: ${MIN_PRICE:,.2f}")
print(f"2. Minimum ADDV: ${MIN_ADDV:,.0f}")
print(f"3. Lookback Period: {LOOKBACK_DAYS} days (~6 months)")
print(f"4. Minimum Market Cap Proxy: ${MIN_MARKET_CAP:,.0f}")
print(f"5. Minimum Data Availability: {MIN_DATA_POINTS} days ({MIN_DATA_POINTS/LOOKBACK_DAYS*100:.0f}%)")

print("\n" + "="*80)
print("UNIVERSE STATISTICS")
print("="*80)
print(f"\nInitial Universe (Russell 2000):")
print(f"  Rows: {initial_count:,}")
print(f"  Unique symbols: {initial_symbols:,}")

print(f"\nEstimation Universe (After Filters):")
print(f"  Rows: {final_count:,}")
print(f"  Unique symbols: {final_symbols:,}")
print(f"  Unique dates: {df_output.index.get_level_values(0).nunique():,}")
print(f"  Date range: {df_output.index.get_level_values(0).min()} to {df_output.index.get_level_values(0).max()}")
print(f"  Average symbols per day: {len(df_output) / df_output.index.get_level_values(0).nunique():.0f}")

print(f"\nReduction:")
print(f"  Rows removed: {total_removed:,} ({total_removed/initial_count*100:.2f}%)")
print(f"  Symbols removed: {initial_symbols - final_symbols:,} ({(initial_symbols - final_symbols)/initial_symbols*100:.2f}%)")

print("\n" + "="*80)
print("UNIVERSE CHARACTERISTICS")
print("="*80)
print(f"\nPrice Range: ${df_output['close'].min():.2f} - ${df_output['close'].max():,.2f}")
print(f"Median Price: ${df_output['close'].median():.2f}")

print(f"\nADDV Range: ${df_output['addv'].min():,.0f} - ${df_output['addv'].max():,.0f}")
print(f"Median ADDV: ${df_output['addv'].median():,.0f}")

print(f"\nMonthly Turnover: {np.mean(turnover_pct):.2f}% average")

print("\n" + "="*80)
print("DATA QUALITY")
print("="*80)
print(f"\n✓ All stocks meet minimum price threshold (${MIN_PRICE:,.2f})")
print(f"✓ All stocks meet minimum liquidity threshold (${MIN_ADDV:,.0f} ADDV)")
print(f"✓ All stocks have sufficient data history ({MIN_DATA_POINTS}+ days)")
print(f"✓ OHLC data consistency validated")

print("\n" + "="*80)
print("OUTPUT")
print("="*80)
print(f"\nFile: {output_path}")
print(f"Size: {file_size_mb:.2f} MB")
print(f"Format: Parquet (MultiIndex: date × symbol)")

print("\n" + "="*80)
print("✅ STEP 2 COMPLETE: ESTIMATION UNIVERSE SELECTED")
print("="*80)
print("\nNext Step: Step 3 - Winsorization")